# MobileNet: Efficient CNNs for Mobile Devices

## Introduction

**MobileNet** (Howard et al., 2017) asked a fundamentally different question than ResNet: **"Can we run CNNs on mobile phones in real-time?"**

While AlexNet, VGG, ResNet focused on **accuracy**, MobileNet optimized for **efficiency**.

**The Problem:**
- ResNet-50: 4 billion FLOPs, 25M parameters
- Mobile phone: 2W power budget, limited memory
- Real-time requirements: 30+ FPS
- Standard convolutions are too expensive!

**The Solution: Depthwise Separable Convolutions**

Factor standard convolution into two operations:
1. **Depthwise:** Filter each channel separately (spatial filtering)
2. **Pointwise:** Combine channels with 1×1 convs (channel mixing)

**Result:** 8-9× fewer operations with minimal accuracy loss!

**What we'll explore:**

- Why standard convolutions are computationally expensive
- Mathematical breakdown of depthwise separable convolutions
- Implement MobileNetV1 architecture
- Compare efficiency vs accuracy trade-offs
- Understand why MobileNet revolutionized mobile vision

**Why this matters:**

MobileNet enabled a new class of applications: real-time vision on **edge devices** (phones, IoT, drones, AR glasses). It's not about pushing accuracy benchmarks - it's about making AI **accessible** and **practical** everywhere.

## 1. Setup

### Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import time

from aiml_notebooks import get_device, set_seed

### Set random seed for reproducibility

In [ ]:
set_seed(42)

### Configure device

In [ ]:
device = get_device()
print(f"Using device: {device}")

## 2. The Computational Cost Problem

### Understanding Standard Convolution Costs

Let's calculate exactly how expensive standard convolutions are.

In [ ]:
def standard_conv_cost(input_channels, output_channels, kernel_size, input_height, input_width):
    """Calculate FLOPs for standard convolution (assuming stride=1, padding to keep size)."""
    # Each output pixel requires kernel_size × kernel_size × input_channels multiplications
    # Plus (kernel_size × kernel_size × input_channels - 1) additions ≈ same count
    # Simplified: 2 × kernel_size² × input_channels operations per output pixel
    
    output_height = input_height  # Assuming padding keeps size
    output_width = input_width
    
    ops_per_output_pixel = kernel_size * kernel_size * input_channels
    total_output_pixels = output_height * output_width * output_channels
    
    total_ops = ops_per_output_pixel * total_output_pixels
    return total_ops

# Example: Common layer in ResNet
C_in = 256
C_out = 256
K = 3
H = 14
W = 14

std_ops = standard_conv_cost(C_in, C_out, K, H, W)

print("Standard 3×3 Convolution Cost:")
print(f"  Input: {H}×{W}×{C_in}")
print(f"  Output: {H}×{W}×{C_out}")
print(f"  Kernel: {K}×{K}")
print(f"  ")
print(f"  Operations: {std_ops:,} multiply-adds")
print(f"             = {std_ops/1e6:.1f} million FLOPs")
print(f"  Parameters: {K}×{K}×{C_in}×{C_out} = {K*K*C_in*C_out:,}")
print(f"\n💡 This is just ONE layer! ResNet-50 has 50+ such layers.")

### Visualize where the cost comes from

In [ ]:
# Show how cost scales with channels and spatial size
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cost vs number of channels
channels = [64, 128, 256, 512, 1024]
costs_channels = [standard_conv_cost(c, c, 3, 14, 14) / 1e6 for c in channels]

ax1.plot(channels, costs_channels, 'o-', linewidth=2.5, markersize=10, color='#FF6B6B')
ax1.set_xlabel('Number of Channels', fontsize=12)
ax1.set_ylabel('FLOPs (Millions)', fontsize=12)
ax1.set_title('Convolution Cost vs Channels (14×14 spatial)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.annotate('Quadratic growth!\nC_in × C_out', 
             xy=(512, costs_channels[3]), xytext=(400, 1500),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, color='red', fontweight='bold')

# Cost vs spatial resolution
resolutions = [7, 14, 28, 56, 112]
costs_spatial = [standard_conv_cost(128, 128, 3, r, r) / 1e6 for r in resolutions]

ax2.plot(resolutions, costs_spatial, 's-', linewidth=2.5, markersize=10, color='#4ECDC4')
ax2.set_xlabel('Spatial Resolution (H=W)', fontsize=12)
ax2.set_ylabel('FLOPs (Millions)', fontsize=12)
ax2.set_title('Convolution Cost vs Resolution (128 channels)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.annotate('Quadratic growth!\nH × W', 
             xy=(56, costs_spatial[3]), xytext=(70, 300),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2),
             fontsize=10, color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️  Standard convolution cost = K² × C_in × C_out × H × W")
print("   This scales VERY badly with channels and resolution!")

## 3. Depthwise Separable Convolutions

### The Key Insight: Separate Spatial and Channel Operations

Standard convolution does two things **simultaneously**:
1. Spatial filtering (apply kernel across space)
2. Channel mixing (combine information across channels)

**MobileNet's idea:** Do them **separately**!

In [ ]:
print("Standard Convolution:")
print("━" * 70)
print("Input: H × W × C_in")
print("Kernel: K × K × C_in × C_out")
print("  ")
print("What it does:")
print("  For each of C_out output channels:")
print("    Apply K×K filter across ALL C_in input channels")
print("    (spatial filtering + channel mixing in one step)")
print("  ")
print("Cost: K² × C_in × C_out × H × W")
print("")
print("="*70)
print("")
print("Depthwise Separable Convolution:")
print("━" * 70)
print("STEP 1: Depthwise Convolution (spatial filtering)")
print("  Input: H × W × C_in")
print("  Kernel: K × K × C_in (separate filter per channel!)")
print("  Output: H × W × C_in")
print("  ")
print("  What it does:")
print("    For each input channel independently:")
print("      Apply K×K filter (only spatial filtering)")
print("      No cross-channel interaction!")
print("  ")
print("  Cost: K² × C_in × H × W")
print("")
print("STEP 2: Pointwise Convolution (channel mixing)")
print("  Input: H × W × C_in")
print("  Kernel: 1 × 1 × C_in × C_out")
print("  Output: H × W × C_out")
print("  ")
print("  What it does:")
print("    At each spatial location:")
print("      Combine all C_in channels to produce C_out channels")
print("      (only channel mixing, no spatial filtering)")
print("  ")
print("  Cost: C_in × C_out × H × W")
print("")
print("Total Cost: K² × C_in × H × W + C_in × C_out × H × W")
print("          = (K² + C_out) × C_in × H × W")

### Calculate the Speedup

In [ ]:
def depthwise_separable_cost(input_channels, output_channels, kernel_size, input_height, input_width):
    """Calculate FLOPs for depthwise separable convolution."""
    # Depthwise: K×K filter per input channel
    depthwise_ops = kernel_size * kernel_size * input_channels * input_height * input_width
    
    # Pointwise: 1×1 conv to mix channels
    pointwise_ops = input_channels * output_channels * input_height * input_width
    
    return depthwise_ops + pointwise_ops

# Same example as before
C_in = 256
C_out = 256
K = 3
H = 14
W = 14

std_ops = standard_conv_cost(C_in, C_out, K, H, W)
sep_ops = depthwise_separable_cost(C_in, C_out, K, H, W)

speedup = std_ops / sep_ops

print("Efficiency Comparison:")
print("="*70)
print(f"Standard Convolution:       {std_ops:>12,} ops ({std_ops/1e6:>6.1f}M)")
print(f"Depthwise Separable:        {sep_ops:>12,} ops ({sep_ops/1e6:>6.1f}M)")
print(f"")
print(f"Speedup: {speedup:.1f}× fewer operations!")
print(f"")
print(f"Theoretical speedup formula:")
print(f"  Standard:  K² × C_in × C_out")
print(f"  Separable: K² × C_in + C_in × C_out = C_in × (K² + C_out)")
print(f"  ")
print(f"  Ratio: [K² × C_in × C_out] / [C_in × (K² + C_out)]")
print(f"       = (K² × C_out) / (K² + C_out)")
print(f"       ≈ C_out / (1 + C_out/K²)  when C_out >> K²")
print(f"       ≈ K²  when C_out >> K²")
print(f"  ")
print(f"For 3×3 convolutions: ~8-9× speedup")
print(f"For 5×5 convolutions: ~25× speedup")

### Visualize the factorization

In [ ]:
# Diagram showing standard vs depthwise separable
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Standard convolution
ax1.text(0.5, 0.9, 'Input\nH×W×C_in', ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
ax1.text(0.5, 0.6, 'Standard Conv\nK×K×C_in×C_out\n(Spatial + Channel)', 
         ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#FF6B6B', alpha=0.7))
ax1.text(0.5, 0.3, 'Output\nH×W×C_out', ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

ax1.annotate('', xy=(0.5, 0.67), xytext=(0.5, 0.85),
            arrowprops=dict(arrowstyle='->', lw=3))
ax1.annotate('', xy=(0.5, 0.35), xytext=(0.5, 0.53),
            arrowprops=dict(arrowstyle='->', lw=3))

ax1.text(0.5, 0.1, f'Cost: K²×C_in×C_out×H×W\n{std_ops/1e6:.1f}M FLOPs', 
         ha='center', fontsize=9, style='italic', color='red')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Standard Convolution', fontsize=13, fontweight='bold')

# Depthwise separable
ax2.text(0.5, 0.9, 'Input\nH×W×C_in', ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
ax2.text(0.5, 0.7, 'Depthwise Conv\nK×K (per channel)\n(Spatial only)', 
         ha='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='#4ECDC4', alpha=0.7))
ax2.text(0.5, 0.5, 'Intermediate\nH×W×C_in', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax2.text(0.5, 0.3, 'Pointwise Conv\n1×1×C_in×C_out\n(Channel only)', 
         ha='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='#45B7D1', alpha=0.7))
ax2.text(0.5, 0.1, 'Output\nH×W×C_out', ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

for y_from, y_to in [(0.85, 0.75), (0.65, 0.55), (0.45, 0.35), (0.25, 0.15)]:
    ax2.annotate('', xy=(0.5, y_to), xytext=(0.5, y_from),
                arrowprops=dict(arrowstyle='->', lw=2))

ax2.text(0.5, 0.02, f'Cost: (K²+C_out)×C_in×H×W\n{sep_ops/1e6:.1f}M FLOPs\n({speedup:.1f}× faster!)', 
         ha='center', fontsize=9, style='italic', color='green', fontweight='bold')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Depthwise Separable Convolution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Key insight: Factor one expensive operation into two cheap operations!")

## 4. Dataset Preparation

### Define data transforms

In [ ]:
# CIFAR-10 normalization statistics
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

### Load CIFAR-10 dataset

In [ ]:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

### Create data loaders

In [ ]:
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

## 5. MobileNetV1 Architecture

### Depthwise Separable Conv Block

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise Separable Convolution block.
    
    Structure:
      1. Depthwise conv (K×K per channel)
      2. BatchNorm + ReLU
      3. Pointwise conv (1×1)
      4. BatchNorm + ReLU
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # Depthwise convolution (groups=in_channels means separate filter per channel)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, 
            kernel_size=3, stride=stride, padding=1, 
            groups=in_channels,  # Key: this makes it depthwise!
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        
        # Pointwise convolution (1×1 to combine channels)
        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1, stride=1, padding=0,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        # Depthwise: spatial filtering per channel
        x = self.depthwise(x)
        x = self.bn1(x)
        x = F.relu(x, inplace=True)
        
        # Pointwise: channel mixing
        x = self.pointwise(x)
        x = self.bn2(x)
        x = F.relu(x, inplace=True)
        
        return x

### MobileNetV1 Full Architecture

In [ ]:
class MobileNetV1(nn.Module):
    """MobileNetV1 architecture.
    
    Args:
        num_classes: Number of output classes
        width_multiplier: Scales number of channels (default 1.0)
    """
    def __init__(self, num_classes=10, width_multiplier=1.0):
        super().__init__()
        
        def _make_divisible(v, divisor=8):
            """Ensure channels are divisible by 8 for efficiency."""
            new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
            if new_v < 0.9 * v:
                new_v += divisor
            return new_v
        
        # Initial standard conv
        self.conv1 = nn.Conv2d(3, _make_divisible(32 * width_multiplier), 
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(_make_divisible(32 * width_multiplier))
        
        # Depthwise separable conv layers
        # Format: (in_channels, out_channels, stride)
        self.layers = nn.ModuleList([
            DepthwiseSeparableConv(_make_divisible(32 * width_multiplier), 
                                   _make_divisible(64 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(64 * width_multiplier), 
                                   _make_divisible(128 * width_multiplier), 2),
            DepthwiseSeparableConv(_make_divisible(128 * width_multiplier), 
                                   _make_divisible(128 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(128 * width_multiplier), 
                                   _make_divisible(256 * width_multiplier), 2),
            DepthwiseSeparableConv(_make_divisible(256 * width_multiplier), 
                                   _make_divisible(256 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(256 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 2),
            # 5 layers of 512 channels
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(512 * width_multiplier), 1),
            DepthwiseSeparableConv(_make_divisible(512 * width_multiplier), 
                                   _make_divisible(1024 * width_multiplier), 2),
            DepthwiseSeparableConv(_make_divisible(1024 * width_multiplier), 
                                   _make_divisible(1024 * width_multiplier), 1),
        ])
        
        # Global average pooling + classifier
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(_make_divisible(1024 * width_multiplier), num_classes)
    
    def forward(self, x):
        # Initial conv
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x, inplace=True)
        
        # Depthwise separable layers
        for layer in self.layers:
            x = layer(x)
        
        # Global average pooling + FC
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

### Inspect MobileNetV1

In [ ]:
model = MobileNetV1(width_multiplier=1.0).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"MobileNetV1 Architecture (width=1.0):")
print(model)
print(f"\nParameter count:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Test forward pass and measure inference time
dummy_input = torch.randn(1, 3, 32, 32).to(device)
output = model(dummy_input)
print(f"\nOutput shape: {output.shape}")

# Warm up
for _ in range(10):
    _ = model(dummy_input)

# Measure inference time
torch.cuda.synchronize() if device.type == 'cuda' else None
start = time.time()
for _ in range(100):
    _ = model(dummy_input)
torch.cuda.synchronize() if device.type == 'cuda' else None
elapsed = time.time() - start

print(f"\nInference time: {elapsed/100*1000:.2f} ms per image")
print(f"Throughput: {100/elapsed:.1f} images/second")

### Compare to ResNet-18

In [ ]:
print("MobileNetV1 vs ResNet-18 Comparison:\n")
print(f"{'Metric':<30} {'ResNet-18':<20} {'MobileNetV1':<20}")
print("─" * 70)
print(f"{'Parameters (CIFAR-10)':<30} {'~11.2M':<20} {f'~{total_params/1e6:.1f}M':<20}")
print(f"{'Primary operation':<30} {'Standard 3×3 conv':<20} {'Depthwise sep ✓':<20}")
print(f"{'Design goal':<30} {'Depth (accuracy)':<20} {'Efficiency ✓':<20}")
print(f"{'FLOPs (relative)':<30} {'~1.0×':<20} {'~0.1-0.2× ✓':<20}")
print(f"{'Mobile deployment':<30} {'Difficult':<20} {'Optimized ✓':<20}")
print("─" * 70)
print(f"\n✅ MobileNet advantages:")
print(f"   • Fewer parameters (~3× reduction)")
print(f"   • Much fewer FLOPs (~8× reduction)")
print(f"   • Designed for mobile/edge devices")
print(f"   • Width multiplier for easy scaling")
print(f"\n⚖️ Trade-off: ~1-3% accuracy drop for massive efficiency gain")

### Width Multiplier: Easy Scaling

In [ ]:
# Show how width multiplier scales the network
widths = [0.25, 0.5, 0.75, 1.0]
sizes = []

for w in widths:
    m = MobileNetV1(width_multiplier=w)
    params = sum(p.numel() for p in m.parameters())
    sizes.append(params)

print("Width Multiplier Scaling:\n")
print(f"{'Width':<10} {'Parameters':<15} {'Relative Size'}")
print("─" * 45)
for w, s in zip(widths, sizes):
    print(f"{w:<10} {s:>12,}   {s/sizes[-1]*100:>5.1f}%")

print(f"\n💡 Width multiplier allows easy accuracy/efficiency trade-off:")
print(f"   • α=1.0: Full model (best accuracy)")
print(f"   • α=0.75: 75% channels (good balance)")
print(f"   • α=0.5: 50% channels (fast inference)")
print(f"   • α=0.25: 25% channels (extreme efficiency)")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar([f'{w}×' for w in widths], [s/1e6 for s in sizes], 
              color=['#FF6B6B', '#FFA07A', '#4ECDC4', '#45B7D1'])
ax.set_xlabel('Width Multiplier', fontsize=12)
ax.set_ylabel('Parameters (Millions)', fontsize=12)
ax.set_title('MobileNetV1 Scaling with Width Multiplier', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, s in zip(bars, sizes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{s/1e6:.1f}M',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Training

### Define training utilities

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{running_loss/len(pbar):.3f}', 
                         'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / len(train_loader), 100. * correct / total

def evaluate(model, test_loader, criterion, device):
    """Evaluate on test set."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(test_loader), 100. * correct / total

### Train MobileNetV1

In [ ]:
# Initialize model (use width=0.75 for faster training)
model = MobileNetV1(width_multiplier=0.75).to(device)

# Training configuration
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=4e-5)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)

# Training history
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

# Training loop
epochs = 20
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

## 7. Results and Analysis

### Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax1.plot(history['test_loss'], label='Test', marker='s', linewidth=2, color='#FF6B6B')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('MobileNetV1 (α=0.75): Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(history['train_acc'], label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax2.plot(history['test_acc'], label='Test', marker='s', linewidth=2, color='#FF6B6B')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('MobileNetV1 (α=0.75): Accuracy', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"  Train Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"  Test Accuracy: {history['test_acc'][-1]:.2f}%")
print(f"  Overfitting Gap: {history['train_acc'][-1] - history['test_acc'][-1]:.2f}%")

## 8. MobileNet's Impact and Legacy

### Real-World Applications

In [ ]:
print("MobileNet's Impact on Production AI:\n")
print("━" * 70)
print("1. MOBILE APPLICATIONS")
print("   • Google Photos: On-device image recognition")
print("   • Snapchat/Instagram: Real-time filters and effects")
print("   • Mobile AR: Object detection for AR experiences")
print("   • Portrait mode: Real-time depth estimation")
print("\n━" * 70)
print("2. EDGE DEVICES")
print("   • Security cameras: Real-time object detection")
print("   • Drones: On-board vision for navigation")
print("   • Robots: Embedded vision systems")
print("   • IoT devices: Smart sensors with vision")
print("\n━" * 70)
print("3. PRIVACY & LATENCY")
print("   • On-device processing (no cloud required)")
print("   • Private - data never leaves device")
print("   • Low latency - no network round-trip")
print("   • Works offline")
print("\n━" * 70)
print("4. COST REDUCTION")
print("   • Cheaper hardware (no GPU needed)")
print("   • Lower power consumption")
print("   • Reduced cloud costs")
print("   • Longer battery life")
print("\n✅ MobileNet democratized AI - from data centers to pockets!")

### Evolution: V1 → V2 → V3

In [ ]:
print("MobileNet Evolution:\n")
print("━" * 70)
print("MobileNetV1 (2017)")
print("  Innovation: Depthwise separable convolutions")
print("  Impact: 8-9× fewer operations than standard conv")
print("  Limitation: Linear bottleneck destroys information")
print("\n━" * 70)
print("MobileNetV2 (2018)")
print("  Innovation: Inverted residuals + linear bottlenecks")
print("  Key insight: ReLU destroys info in low-dim space")
print("  Solution: Expand → Depthwise → Project (no ReLU on projection)")
print("  Impact: Better accuracy at same cost")
print("\n━" * 70)
print("MobileNetV3 (2019)")
print("  Innovation: Neural Architecture Search (NAS)")
print("  Additions: Squeeze-and-Excitation, h-swish activation")
print("  Impact: State-of-the-art efficiency/accuracy trade-off")
print("\n━" * 70)
print("\nCommon theme: Maximize accuracy per FLOP")
print("V1 proved the concept, V2/V3 refined the details.")

### Comparison with Other Efficient Architectures

In [ ]:
# Visualize the efficiency/accuracy landscape
fig, ax = plt.subplots(figsize=(12, 7))

# Approximate ImageNet top-1 accuracy vs FLOPs (in millions)
# Format: (FLOPs, Accuracy, Name)
architectures = [
    (4000, 76.1, 'ResNet-50'),
    (7800, 77.4, 'ResNet-101'),
    (11300, 77.6, 'ResNet-152'),
    (15500, 77.3, 'VGG-16'),
    (5700, 78.0, 'Inception-v3'),
    (569, 70.6, 'MobileNetV1'),
    (300, 72.0, 'MobileNetV2'),
    (219, 75.2, 'MobileNetV3-Large'),
    (150, 67.4, 'MobileNetV3-Small'),
    (325, 69.4, 'ShuffleNetV2'),
    (390, 77.1, 'EfficientNet-B0'),
]

# Separate by family
standard = [(x[0], x[1], x[2]) for x in architectures if x[2].startswith(('ResNet', 'VGG', 'Inception'))]
mobile = [(x[0], x[1], x[2]) for x in architectures if x[2].startswith('Mobile')]
efficient = [(x[0], x[1], x[2]) for x in architectures if x[2].startswith(('Shuffle', 'Efficient'))]

# Plot
for arch_list, color, label in [
    (standard, '#FF6B6B', 'Standard CNNs'),
    (mobile, '#4ECDC4', 'MobileNets'),
    (efficient, '#FFA07A', 'Other Efficient')
]:
    flops = [x[0] for x in arch_list]
    acc = [x[1] for x in arch_list]
    ax.scatter(flops, acc, s=150, alpha=0.7, color=color, label=label, edgecolors='black', linewidth=1.5)
    
    for f, a, name in arch_list:
        ax.annotate(name, (f, a), xytext=(5, 5), textcoords='offset points', 
                   fontsize=9, alpha=0.8)

ax.set_xlabel('FLOPs (Millions)', fontsize=13, fontweight='bold')
ax.set_ylabel('ImageNet Top-1 Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('Efficiency vs Accuracy: MobileNet\'s Sweet Spot', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

# Draw Pareto frontier approximation
ax.axvline(x=600, color='green', linestyle='--', alpha=0.5, linewidth=2)
ax.text(650, 64, 'Mobile/Edge\nRegion', fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 MobileNets occupy a unique efficiency region:")
print("   • 10-50× fewer FLOPs than standard CNNs")
print("   • Only 1-4% accuracy drop")
print("   • Enables real-time mobile deployment")

## 9. Summary: The Efficiency Revolution

### What MobileNet Achieved

| Achievement | Impact |
|-------------|--------|
| **Depthwise separable convs** | 8-9× fewer operations |
| **Width multiplier** | Easy accuracy/speed trade-off |
| **Mobile deployment** | Real-time vision on phones |
| **Privacy** | On-device processing |
| **Accessibility** | AI everywhere, not just data centers |

### The Dual Evolution of CNNs

After ResNet (2015), CNNs evolved in two parallel directions:

```
             ResNet (2015)
                  |
        ┌─────────┴─────────┐
        │                   │
   ACCURACY              EFFICIENCY
        │                   │
    DenseNet           MobileNetV1 (2017)
    SENet              MobileNetV2 (2018)
    ResNeXt            MobileNetV3 (2019)
                       ShuffleNet
                       SqueezeNet
        │                   │
        └─────────┬─────────┘
                  │
           EfficientNet (2019)
         (Optimal scaling of both)
```

### Key Lessons

1. **Efficiency matters as much as accuracy** for real-world deployment
2. **Factorization is powerful** - separate spatial and channel operations
3. **Constraints drive innovation** - mobile limitations → new architectures
4. **Trade-offs are acceptable** - small accuracy drop for huge efficiency gain
5. **Accessibility is impact** - AI on billions of devices > best benchmark score

### Modern Legacy

MobileNet's principles influence modern architectures:

- **Transformers:** Factorized attention (separate Q, K, V projections)
- **Diffusion models:** Efficient U-Nets with depthwise convs
- **Edge AI:** Entire field of on-device ML
- **TinyML:** Sub-MB models for microcontrollers

---

### Final Thought

MobileNet showed that **architectural innovation isn't just about pushing SOTA** - it's about making AI **practical, accessible, and ubiquitous**.

The question shifted from:
- "How accurate can we get?" (ResNet, DenseNet)

To:
- "How efficient can we be?" (MobileNet, ShuffleNet)

And finally:
- "What's the optimal balance?" (EfficientNet, NAS)

This completes the CNN efficiency story. The next frontier: **Transformers** and the shift from convolutional inductive bias to learned attention!